In [2]:
import os
import time
import subprocess
import pandas as pd

from pathlib import Path
from tqdm import tqdm
from PIL import Image

In [3]:
def compressImg(input_path, output_path):
    """
    Compress an img using WebP.
    """
    cmd = ['cwebp', '-lossless', '-o', str(output_path), str(input_path)]
    try:
        subprocess.run(cmd, check=True)
    except Exception as e:
        print(f'Error compressing {input_path}: {e}.')
        return False
    return True

def decompressImg(input_path, output_path):
    """
    DeCompress an WebP img.
    """
    cmd = ['dwebp', str(input_path), '-o', str(output_path)]
    try:
        subprocess.run(cmd, check=True)
    except Exception as e:
        print(f'Error decompressing {input_path}: {e}.')
        return False
    return True

In [4]:
def cal_compression_ratio(original_path, compressed_path):
    """
    Calculate compression rate based on file sizes.
    """
    original_size = os.path.getsize(original_path)
    compressed_size = os.path.getsize(compressed_path)

    with Image.open(original_path) as img:
        width, height = img.size
        npixels = width * height
    bpsp = (compressed_size * 8) / (npixels * 3)
    compression_ratio = compressed_size / original_size
    
    return compression_ratio, original_size, compressed_size, bpsp, width, height, npixels

In [5]:
def process_dataset(dataset_path, output_dir, mode="compress", cal_metric=True, verbose=False):
    """
    Process a dataset (Kodak or CLIC) using WebP, with optional compression rate calculation.
    """
    dataset_path = Path(dataset_path)
    output_dir   = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    imgs = list(dataset_path.glob('*.png'))
    if len(imgs) == 0:
        print(f'No Imgs found in {dataset_path}.')
        return
    
    rows = []
    for img in imgs:
        output_path = output_dir / f'{img.stem}.webp' if mode == 'compress' else output_dir / f'{img.stem}_recon.png'
        
        start = time.time()
        if mode == 'compress':
            success = compressImg(img, output_path)
        elif mode == 'decompress':
            success = decompressImg(img, output_path)
        elapsed = time.time() - start   
          
        if success and cal_metric and mode == 'compress':
            compression_ratio, original_size, compressed_size, bpsp, width, height, npixels = cal_compression_ratio(img, output_path)
            if verbose:
                print(f'{img.name}: compression ratio = {compression_ratio:.2%} (Original: {original_size} bytes, Compressed: {compressed_size} bytes), bpsp: {bpsp}.')
            rows.append([img.name, bpsp, compression_ratio, original_size, compressed_size, width, height, npixels, elapsed])
            
    return rows

In [6]:
""" 1. compress Touch and Go (png) """
dataset_path = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/compressed/webp'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/webp_TouchandGoDataset-v2.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/webp_TouchandGoDataset-v2.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,1400.000000,1400.000000,1400.000000,1400.000000,1400.0,1400.0,1400.0,1400.000000,1400.000000
mean,0.936295,0.371137,288034.526429,107861.197143,640.0,480.0,307200.0,0.224848,2.500300
std,0.217291,0.033322,44205.507025,25031.976118,0.0,0.0,0.0,0.016051,0.383728
min,0.427431,0.255994,192348.000000,49240.000000,640.0,480.0,307200.0,0.180235,1.669687
25%,0.793798,0.350209,253517.500000,91445.500000,640.0,480.0,307200.0,0.214199,2.200673
50%,0.891207,0.367601,276483.000000,102667.000000,640.0,480.0,307200.0,0.221490,2.400026
75%,0.992253,0.391384,310647.000000,114307.500000,640.0,480.0,307200.0,0.233160,2.696589
max,1.943750,0.485732,481097.000000,223920.000000,640.0,480.0,307200.0,0.293930,4.176189


In [7]:
""" 2. compress Objectfolder (png) """
dataset_path = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/compressed/webp'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/webp_ObjectFolder_1.0.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/webp_ObjectFolder_1.0.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,20000.000000,20000.000000,20000.000000,20000.000000,20000.0,20000.0,20000.0,20000.000000,20000.000000
mean,3.611821,0.910498,28541.906000,26005.113900,160.0,120.0,19200.0,0.034075,3.964154
std,0.370934,0.011567,2695.165442,2670.728183,0.0,0.0,0.0,0.005515,0.374329
min,2.992778,0.851440,24187.000000,21548.000000,160.0,120.0,19200.0,0.024899,3.359306
25%,3.337986,0.902891,26490.000000,24033.500000,160.0,120.0,19200.0,0.028676,3.679167
50%,3.489167,0.909593,27698.000000,25122.000000,160.0,120.0,19200.0,0.034672,3.846944
75%,3.837847,0.917196,30118.000000,27632.500000,160.0,120.0,19200.0,0.038007,4.183056
max,5.507778,0.956821,41962.000000,39656.000000,160.0,120.0,19200.0,0.100839,5.828056


In [8]:
""" 3. compress SSVTP (png) """
dataset_path = '/data/ssd/zhaoy/datasets/SSVTP/dataset-comp/test'
output_dir = '/data/ssd/zhaoy/datasets/SSVTP/compressed/webp'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/webp_SSVTP.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/webp_SSVTP.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,918.000000,918.000000,918.000000,918.000000,918.0,918.0,918.0,918.000000,918.000000
mean,1.820040,0.815499,64301.678649,52417.156863,240.0,320.0,76800.0,0.080829,2.232697
std,0.129281,0.010789,4853.413152,3723.304495,0.0,0.0,0.0,0.009192,0.168521
min,1.399167,0.739878,53899.000000,40296.000000,240.0,320.0,76800.0,0.072541,1.871493
25%,1.715451,0.811792,60208.000000,49405.000000,240.0,320.0,76800.0,0.075882,2.090556
50%,1.805903,0.817537,63670.500000,52010.000000,240.0,320.0,76800.0,0.077226,2.210781
75%,1.917083,0.821806,67915.000000,55212.000000,240.0,320.0,76800.0,0.084260,2.358160
max,2.152569,0.833506,77399.000000,61994.000000,240.0,320.0,76800.0,0.181998,2.687465


In [9]:
""" 4. compress ObjTac (png) """
dataset_path = '/data/ssd/zhaoy/datasets/ObjTac/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/ObjTac/compressed/webp'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/webp_ObjTac.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/webp_ObjTac.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,51.000000,51.000000,51.000000,51.00000,51.0,51.000000,51.000000,51.000000,51.000000
mean,0.424436,0.630694,11784.803922,8683.72549,60.0,903.352941,54201.176471,0.098939,0.578609
std,0.293211,0.206105,7995.848535,6714.16026,0.0,391.917891,23515.073434,0.060547,0.328989
min,0.009479,0.149144,818.000000,122.00000,60.0,331.000000,19860.000000,0.013375,0.063559
25%,0.128042,0.471742,4868.500000,2002.00000,60.0,587.500000,35250.000000,0.044550,0.253969
50%,0.448312,0.717236,10936.000000,8148.00000,60.0,813.000000,48780.000000,0.098999,0.629227
75%,0.653114,0.785750,19075.500000,14788.00000,60.0,1142.000000,68520.000000,0.146872,0.826285
max,1.056076,0.859354,28278.000000,21374.00000,60.0,1802.000000,108120.000000,0.271438,1.246181


In [11]:
""" 5. compress YCB-Slide (png) """
dataset_path = '/data/ssd/zhaoy/datasets/YCB-Slide/dataset-comp/test/image'
output_dir = '/data/ssd/zhaoy/datasets/YCB-Slide/compressed/webp'

# rows = process_dataset(dataset_path, output_dir, mode='compress', verbose=False)
# df = pd.DataFrame(rows, columns=['img', 'bpsp', 'compression_ratio', 'original_size', 'compressed_size', 'width', 'height', 'npixels', 'elapsed'])

# df['original_bpsp'] = (df['original_size'] * 8) / (df['npixels'] * 3)       # bpsp of png
# display(df)
# display(df.describe())
# df.to_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/webp_YCB-Slide.csv', index=False)

display(pd.read_csv('/home/zhaoy/TaCo-Bench/lossless/statistics/webp_YCB-Slide.csv').describe())

,bpsp,compression_ratio,original_size,compressed_size,width,height,npixels,elapsed,original_bpsp
count,10916.000000,10916.000000,10916.000000,10916.000000,10916.0,10916.0,10916.0,10916.000000,10916.000000
mean,1.767460,0.809722,62881.925247,50902.837120,240.0,320.0,76800.0,0.079580,2.183400
std,0.069608,0.009654,2789.584906,2004.706103,0.0,0.0,0.0,0.004890,0.096861
min,1.553681,0.740407,54243.000000,44746.000000,240.0,320.0,76800.0,0.070194,1.883438
25%,1.716944,0.804532,60928.750000,49448.000000,240.0,320.0,76800.0,0.074757,2.115582
50%,1.759167,0.811386,62656.000000,50664.000000,240.0,320.0,76800.0,0.081103,2.175556
75%,1.810781,0.816201,64545.250000,52150.500000,240.0,320.0,76800.0,0.083275,2.241155
max,2.030208,0.835211,73586.000000,58470.000000,240.0,320.0,76800.0,0.117895,2.555069
